# Week 4, Notebook 3: Predicting Stock Prices

**A neural network takes on the Jamaica Stock Exchange.**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A neural network that predicts the **next-day price** of a Caribbean stock.
- The trick that turns a **time series** into something a network can learn from.
- An honest **baseline** that is surprisingly hard to beat, and the lesson hidden
  in that fact.

### The big idea
We will predict **Blue Mountain Holdings (ticker BMH)**, a company listed on the
Jamaica Stock Exchange. (The data is realistic but made-up, so no one loses real
money while we learn.)

Markets are close to **random** from day to day. That makes this the perfect
place to learn a humbling truth of AI: *a fancy model is worthless if it cannot
beat a dumb baseline.* We will hold the network to that standard.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

stock = pd.read_csv("data/jse_stock.csv", parse_dates=["date"])
print("Trading days:", len(stock))
stock.head()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(stock["date"], stock["price"])
plt.title("Blue Mountain Holdings (BMH) - Jamaica Stock Exchange, daily price")
plt.xlabel("date"); plt.ylabel("price (J$)")
plt.grid(alpha=0.3)
plt.show()

### Step 1: Turn a time series into a learning problem

A neural network needs rows of *inputs* and an *answer*. A price chart is just
one long line, so we reshape it: to predict a given day's price, we feed the
network the **prices from the 5 days before it**. Each such window becomes one
training row. This "sliding window" is the standard way to hand time series to a
neural network.

In [ ]:
prices = stock["price"].values
window = 5    # use the last 5 days to predict the next day

X, y = [], []
for i in range(window, len(prices)):
    X.append(prices[i-window:i])   # the 5 days before
    y.append(prices[i])            # the day we want to predict
X = np.array(X)
y = np.array(y)

print("Example input (5 previous prices):", X[0].round(2))
print("The price to predict next:        ", round(y[0], 2))
print("Total learning rows:", len(X))

### Step 2: Split by time, never shuffle

For a time series you must respect the calendar: train on the **past**, test on
the **future**. Shuffling would let the network peek at future days, which is
cheating you could never do in real life. So we cut at 80% and keep the order.

In [ ]:
from sklearn.preprocessing import StandardScaler

cut = int(len(X) * 0.8)
X_train, X_test = X[:cut], X[cut:]
y_train, y_test = y[:cut], y[cut:]

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train days:", len(X_train), " Test days:", len(X_test))

### Step 3: The baseline every forecaster must beat

The simplest possible forecast: **"tomorrow's price equals today's price."** No
model, no training, just repeat the last known price. It sounds silly, but for a
near-random walk it is shockingly good. This is our line in the sand.

We measure error with **MAE** (mean absolute error): on average, how many dollars
off is the forecast?

In [ ]:
from sklearn.metrics import mean_absolute_error

# Baseline: predict the price = yesterday's price (the last value in the window).
baseline_pred = X_test[:, -1]
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline MAE (predict today = yesterday): J${baseline_mae:.3f}")

### Step 4: Build and train the neural network

A small deep network: two hidden layers, and a single output neuron with **no
squashing** (a price can be any positive number, not a 0-to-1 probability). The
loss is **mse** (mean squared error), the natural score for predicting a number.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(0)

model = keras.Sequential([
    layers.Input(shape=(window,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),                 # one number out: the predicted price
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(X_train_s, y_train,
                    validation_split=0.15,
                    epochs=60, batch_size=16, verbose=0)

plt.figure(figsize=(7, 4))
plt.plot(history.history["mae"], label="training")
plt.plot(history.history["val_mae"], label="validation")
plt.title("Learning curve (MAE, lower is better)")
plt.xlabel("epoch"); plt.ylabel("mean absolute error"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Step 5: Did the network beat the baseline?

Now the moment of truth. We compare the network's error on the unseen test days
against the "predict yesterday" baseline.

In [ ]:
nn_pred = model.predict(X_test_s, verbose=0).flatten()
nn_mae = mean_absolute_error(y_test, nn_pred)

print(f"Baseline MAE: J${baseline_mae:.3f}")
print(f"Network MAE:  J${nn_mae:.3f}")
gap = (baseline_mae - nn_mae) / baseline_mae * 100
print(f"\nThe network is {gap:+.1f}% vs the baseline.")
print("Barely different - and that is the whole lesson." )

In [ ]:
# See the predictions tracking the real price on the test period.
test_dates = stock["date"].values[window:][cut:]
plt.figure(figsize=(11, 4))
plt.plot(test_dates, y_test, label="real price", linewidth=2)
plt.plot(test_dates, nn_pred, label="network prediction", alpha=0.8)
plt.title("BMH: network prediction vs reality (unseen test period)")
plt.xlabel("date"); plt.ylabel("price (J$)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Why so close? Because prices are nearly a random walk

The network's prediction basically **copies yesterday's price**, because that
really is the best guess for a random walk. It did not fail; it discovered the
truth about markets. The value moves so unpredictably day to day that no amount
of network depth extracts a reliable signal from price alone.

This is the single most important lesson in applied AI:

> **The edge is in the data, not the model.** A predictable problem yields to a
> simple model. A near-random one resists even a deep network. Knowing which is
> which is the real skill.

Compare that with the oil, GDP, and crime notebooks coming next, where the data
*does* carry a strong signal and the network genuinely shines.

### A fairer question: will the price go up or down tomorrow?

Predicting the exact price is hard. Predicting just the **direction** (up or
down) sounds easier. Let us test that too, and stay honest with a baseline of
"always guess the more common direction".

In [ ]:
# Direction target: 1 if next day's price is higher than today's, else 0.
returns = np.diff(prices) / prices[:-1]
dir_X, dir_y = [], []
for i in range(window, len(returns)):
    dir_X.append(returns[i-window:i])       # last 5 daily returns
    dir_y.append(1 if returns[i] > 0 else 0)
dir_X, dir_y = np.array(dir_X), np.array(dir_y)

c = int(len(dir_X) * 0.8)
dxt, dxe = dir_X[:c], dir_X[c:]
dyt, dye = dir_y[:c], dir_y[c:]

dir_model = keras.Sequential([
    layers.Input(shape=(window,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
dir_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
dir_model.fit(dxt, dyt, epochs=40, batch_size=16, verbose=0)

_, acc = dir_model.evaluate(dxe, dye, verbose=0)
dir_baseline = max(dye.mean(), 1 - dye.mean())
print(f"Direction accuracy: {acc:.1%}")
print(f"Baseline (always guess majority): {dir_baseline:.1%}")
print("\nBarely above a coin flip. The market keeps its secrets.")

### What you learned

- Turn a **time series** into a learning problem with a **sliding window** of past
  values.
- Split time series **by date**, never by shuffling, or you cheat by peeking at
  the future.
- Judge a price model against the **"predict yesterday" baseline**, and a
  direction model against the **majority-guess baseline**.
- Real markets are close to a **random walk**, so even a deep network barely
  beats the baseline. That is not a bug; it is the lesson.

### Try it yourself
1. Change `window` to 10 or 20 days. Does more history help? (Usually not much.)
2. Add `volume` as an extra input alongside the prices.
3. Make the network much bigger. Notice it *still* cannot beat the baseline. That
   stubborn result is the point.